# 05 — Compare on identical page-level candidates
Both heads score the saved validation texts and every matched character on each page. Top-1 speaker accuracy is conditional on successful text/speaker matching; coverage is shown alongside it. Non-nearest and close-wrong-nearest subsets are defined by GT pixel boxes among the same matched candidates. Close means center distance / page diagonal ≤ 0.10.

Single-candidate texts are counted separately because Top-1 is trivial there. Validation also selects the checkpoint, so these results are development measurements rather than an independent test-set estimate.


In [ ]:
import sys
from pathlib import Path

ROOT = Path(r"C:\Users\ESHAAN\BYTE\Applied-Task\Magi\Manga109Association")
sys.path.insert(0, str(ROOT))
from common import *

DATA.mkdir(exist_ok=True)
CHECKPOINTS.mkdir(exist_ok=True)


In [ ]:
import copy
import random
import matplotlib.pyplot as plt
import pandas as pd

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
val_manifest = load_manifest("val")
extraction = read_json(DATA / "extraction_manifest.json")
assert extraction["val_pages_hash"] == digest(read_json(DATA / "val_pages.json"))
baseline_saved = read_json(DATA / "baseline.json")
original = torch.load(CHECKPOINTS / "original_head.pt", map_location="cpu", weights_only=True)
best = torch.load(CHECKPOINTS / "best_association_head.pt", map_location="cpu", weights_only=True)
assert original["extraction_hash"] == best["extraction_hash"] == baseline_saved["extraction_hash"] == digest(extraction)
assert best["candidate_hash"] == baseline_saved["candidate_hash"] == val_manifest["candidate_hash"]

model = load_magi("cpu")
head = copy.deepcopy(model.text_character_matching_head).to(DEVICE)
del model
head.load_state_dict(original["state_dict"], strict=True)
baseline_metrics, baseline_rows, baseline_scores = evaluate(head, val_manifest, DEVICE, best["metrics"]["close_threshold"])
head.load_state_dict(best["state_dict"], strict=True)
finetuned_metrics, finetuned_rows, finetuned_scores = evaluate(head, val_manifest, DEVICE, best["metrics"]["close_threshold"])
keys = lambda rows: [(r["cache_file"], r["text_id"], r["truth"], r["candidate_ids"]) for r in rows]
assert keys(baseline_rows) == keys(finetuned_rows) == keys(baseline_saved["rows"])
display(pd.DataFrame([baseline_metrics, finetuned_metrics], index=["BASELINE", "FINETUNED"]).T)
display(pd.DataFrame(read_json(DATA / "coverage_summary.json")))
save_json(DATA / "evaluation.json", {
    "extraction_hash": digest(extraction), "candidate_hash": val_manifest["candidate_hash"],
    "baseline": baseline_metrics, "finetuned": finetuned_metrics,
    "baseline_rows": baseline_rows, "finetuned_rows": finetuned_rows,
})


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3))
for ax, name, scores in [(axes[0], "BASELINE", baseline_scores), (axes[1], "FINETUNED", finetuned_scores)]:
    for label, color in [(0, "orange"), (1, "green")]:
        values = [p for y, p in zip(scores["labels"], scores["probabilities"]) if y == label]
        if values:
            ax.hist(values, bins=30, range=(0, 1), alpha=0.5, density=True, color=color, label="positive" if label else "negative")
    ax.set_title(name)
    ax.set_xlabel("Pair sigmoid score")
    ax.legend()
plt.show()


## Show about 20 examples
Prefer cases with a nearby wrong character and a farther true speaker, followed by other non-nearest cases. Each plot shows GT, original prediction, and fine-tuned prediction. The paired sampling is seeded and does not alter metrics.


In [ ]:
paired = list(zip(baseline_rows, finetuned_rows))
random.Random(SEED).shuffle(paired)
paired.sort(key=lambda pair: (not pair[0]["close_wrong_nearest"], not pair[0]["non_nearest"]))
records = {p["cache_file"]: p for p in val_manifest["pages"]}
for before, after in paired[:20]:
    page = load_cached(records[before["cache_file"]])
    print("Text:", before["text_id"], "GT:", before["truth"],
          "BASELINE:", before["prediction"], "FINETUNED:", after["prediction"],
          "Non-nearest:", before["non_nearest"])
    show_page(page, before["text_id"], [("BASELINE", before["prediction"], "red"), ("FINETUNED", after["prediction"], "purple")])

overall_improved = finetuned_metrics["top1"] > baseline_metrics["top1"]
before_non_nearest = baseline_metrics["non_nearest_top1"]
after_non_nearest = finetuned_metrics["non_nearest_top1"]
non_nearest_improved = before_non_nearest is not None and after_non_nearest > before_non_nearest
print("Overall improved:", overall_improved)
print("Non-Nearest Speaker Accuracy improved:", non_nearest_improved)
print("Experiment 1 success criterion met:", overall_improved and non_nearest_improved)
